In [ ]:
!pip install transformers torch scikit-learn pandas numpy matplotlib seaborn -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import torch
import re
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             roc_curve)

DATA_DIR  = '/content/drive/MyDrive/CryptoGuard/data'
FIG_DIR   = f'{DATA_DIR}/outputs/figures'
os.makedirs(FIG_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
import re

def clean_text(text):
    """Standardised preprocessing pipeline which is identical to 02_preprocessing."""
    if not isinstance(text, str): return ''
    text = re.sub(r'<[^>]+>', ' ', text)        # strip HTML
    text = re.sub(r'http\S+|www\S+', ' ', text) # remove URLs
    text = re.sub(r'\S+@\S+', ' ', text)         # remove email addresses
    text = re.sub(r'\s+', ' ', text).strip()      # normalise whitespace
    return text.lower()

#general email dataset (pre-split by 02_preprocessing.ipynb)
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
val_df   = pd.read_csv(f'{DATA_DIR}/val.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

print(f"General email dataset:")
print(f"  Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"  Train label distribution: {train_df['label'].value_counts().to_dict()}")

#blockchain synthetic dataset
from sklearn.model_selection import train_test_split

bc_df = pd.read_csv(f'{DATA_DIR}/blockchain_synthetic_clean.csv')
bc_df['text'] = bc_df['text'].apply(clean_text)
bc_df = bc_df[bc_df['text'].str.len() > 20].copy()

print(f"\nBlockchain synthetic dataset:")
print(f"  Total: {len(bc_df)} | Label distribution: {bc_df['label'].value_counts().to_dict()}")

bc_train, bc_test = train_test_split(
    bc_df, test_size=0.15, random_state=42, stratify=bc_df['label'])
print(f"  Train: {len(bc_train)} | Test: {len(bc_test)}")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast

class EmailDataset(Dataset):
    """PyTorch Dataset wrapper for tokenised text classification data."""
    def __init__(self, texts, labels, tokeniser, max_len=256):
        self.encodings = tokeniser(
            list(texts),
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors='pt'
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx]
        }

#load tokeniser from HuggingFace hub
tokeniser = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

#general email datasets
train_dataset = EmailDataset(train_df['text'], train_df['label'], tokeniser)
val_dataset   = EmailDataset(val_df['text'],   val_df['label'],   tokeniser)
test_dataset  = EmailDataset(test_df['text'],  test_df['label'],  tokeniser)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
from transformers import (DistilBertForSequenceClassification,
                          get_linear_schedule_with_warmup)
from torch.optim import AdamW

# Initialise pre-trained DistilBERT with 2-class classification head
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2)
model.to(device)

EPOCHS = 3
LR     = 2e-5

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=100,
    num_training_steps=len(train_loader) * EPOCHS
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training on {device} for {EPOCHS} epochs...")

train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    # ── Training ──────────────────────────────────────────────────────────
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            labels=batch['labels'].to(device)
        )
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += outputs.loss.item()

    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ── Validation ────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            outputs = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
                labels=batch['labels'].to(device)
            )
            val_loss += outputs.loss.item()

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

# Save checkpoint
model.save_pretrained(f'{DATA_DIR}/model_checkpoint')
tokeniser.save_pretrained(f'{DATA_DIR}/model_checkpoint')
print(f"\nGeneral model saved to {DATA_DIR}/model_checkpoint")

In [ ]:
def bert_predict(m, texts, tokeniser, device, batch_size=32):
    """Run inference on a list of texts, return predictions and probabilities."""
    preds, probs = [], []
    m.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = list(texts[i:i+batch_size])
            inputs = tokeniser(
                batch_texts, truncation=True, padding=True,
                max_length=256, return_tensors='pt').to(device)
            outputs = m(**inputs)
            p = torch.softmax(outputs.logits, dim=1)[:,1]
            preds.extend((p >= 0.5).long().cpu().numpy())
            probs.extend(p.cpu().numpy())
    return np.array(preds), np.array(probs)

#evaluate on general email test set
bert_preds, bert_probs = bert_predict(model, test_df['text'].values, tokeniser, device)
y_test = test_df['label'].values

print("=== DistilBERT General — Email Test Set (n=1,600) ===")
print(f"Accuracy:  {accuracy_score(y_test, bert_preds):.4f}")
print(f"Precision: {precision_score(y_test, bert_preds):.4f}")
print(f"Recall:    {recall_score(y_test, bert_preds):.4f}")
print(f"F1:        {f1_score(y_test, bert_preds):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, bert_probs):.4f}")

#training loss curve
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(range(1, EPOCHS+1), train_losses, 'b-o', label='Training Loss', linewidth=2)
ax.plot(range(1, EPOCHS+1), val_losses,   'r-s', label='Validation Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Figure 4.4: DistilBERT Training and Validation Loss')
ax.legend()
ax.set_xticks(range(1, EPOCHS+1))
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig4_4_training_loss.png', dpi=150)
plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

#retrain TF-IDF on general training data
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train_df['text'])
X_test_tfidf  = tfidf.transform(test_df['text'])

lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train_tfidf, train_df['label'])

tfidf_preds = lr.predict(X_test_tfidf)
tfidf_probs = lr.predict_proba(X_test_tfidf)[:,1]

print("=== TF-IDF Baseline — Email Test Set (n=1,600) ===")
print(f"Accuracy:  {accuracy_score(y_test, tfidf_preds):.4f}")
print(f"Precision: {precision_score(y_test, tfidf_preds):.4f}")
print(f"Recall:    {recall_score(y_test, tfidf_preds):.4f}")
print(f"F1:        {f1_score(y_test, tfidf_preds):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, tfidf_probs):.4f}")

In [ ]:
plt.rcParams.update({'font.size': 11, 'figure.dpi': 150})

#Fig 4.1 — TF-IDF confusion matrix
fig, ax = plt.subplots(figsize=(5,4))
cm = confusion_matrix(y_test, tfidf_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate','Phishing'],
            yticklabels=['Legitimate','Phishing'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Figure 4.1: TF-IDF Baseline\nGeneral Test Set (n=1,600)')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig4_1_tfidf_confusion.png', dpi=150)
plt.show()

#Fig 4.2 — DistilBERT confusion matrix
fig, ax = plt.subplots(figsize=(5,4))
cm = confusion_matrix(y_test, bert_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate','Phishing'],
            yticklabels=['Legitimate','Phishing'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Figure 4.2: DistilBERT General\nGeneral Test Set (n=1,600)')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig4_2_bert_confusion.png', dpi=150)
plt.show()

#Fig 4.3 — ROC curves overlay
fig, ax = plt.subplots(figsize=(7,5))
for name, probs, labels in [
    ('TF-IDF + LR', tfidf_probs, y_test),
    ('DistilBERT (general)', bert_probs, y_test),
]:
    fpr, tpr, _ = roc_curve(labels, probs)
    auc = roc_auc_score(labels, probs)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})', linewidth=2)
ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Figure 4.3: ROC Curves — General Email Test Set')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig4_3_roc_curves.png', dpi=150)
plt.show()

In [ ]:
#blockchain test set predictions — general models
X_bc_tfidf = tfidf.transform(bc_test['text'])
tfidf_bc_preds = lr.predict(X_bc_tfidf)
bert_bc_preds, _ = bert_predict(model, bc_test['text'].values, tokeniser, device)
y_bc = bc_test['label'].values

print("DOMAIN GAP ANALYSIS")
print(f"{'Model':<35} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6}")
print("-"*58)

for name, preds in [
    ("TF-IDF + LR (blockchain)",      tfidf_bc_preds),
    ("DistilBERT general (blockchain)", bert_bc_preds),
]:
    print(f"{name:<35} "
          f"{accuracy_score(y_bc, preds):>6.4f} "
          f"{precision_score(y_bc, preds, zero_division=0):>6.4f} "
          f"{recall_score(y_bc, preds, zero_division=0):>6.4f} "
          f"{f1_score(y_bc, preds, zero_division=0):>6.4f}")

print()
print("Key finding: Both models exhibit substantial performance degradation")
print("on blockchain communications compared to the general email test set.")
print(f"DistilBERT F1 drops from 0.9869 (general) to "
      f"{f1_score(y_bc, bert_bc_preds, zero_division=0):.4f} (blockchain)")
print("This constitutes the domain gap this project addresses.")

In [ ]:
#blockchain datasets
bc_train_dataset = EmailDataset(bc_train['text'], bc_train['label'], tokeniser)
bc_test_dataset  = EmailDataset(bc_test['text'],  bc_test['label'],  tokeniser)
bc_train_loader  = DataLoader(bc_train_dataset, batch_size=16, shuffle=True)
bc_test_loader   = DataLoader(bc_test_dataset,  batch_size=16, shuffle=False)

print(f"Blockchain train batches: {len(bc_train_loader)}")
print(f"Blockchain test batches:  {len(bc_test_loader)}")

#load fresh copy of general model for adaptation
from transformers import DistilBertForSequenceClassification as DBSC
adapted_model = DBSC.from_pretrained(f'{DATA_DIR}/model_checkpoint')
adapted_model.to(device)

ADAPT_EPOCHS = 5
ADAPT_LR     = 1e-5

adapt_optimizer = AdamW(adapted_model.parameters(), lr=ADAPT_LR, weight_decay=0.01)
adapt_scheduler = get_linear_schedule_with_warmup(
    adapt_optimizer,
    num_warmup_steps=len(bc_train_loader),
    num_training_steps=len(bc_train_loader) * ADAPT_EPOCHS
)

adapt_train_losses, adapt_val_f1s = [], []
best_f1, best_epoch = 0, 0

print(f"\nDomain adaptation: {ADAPT_EPOCHS} epochs at LR={ADAPT_LR}")
print(f"Training samples: {len(bc_train)} | Test samples: {len(bc_test)}")
print()

for epoch in range(ADAPT_EPOCHS):
    #training
    adapted_model.train()
    total_loss = 0
    for batch in bc_train_loader:
        adapt_optimizer.zero_grad()
        outputs = adapted_model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            labels=batch['labels'].to(device)
        )
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(adapted_model.parameters(), 1.0)
        adapt_optimizer.step()
        adapt_scheduler.step()
        total_loss += outputs.loss.item()

    avg_loss = total_loss / len(bc_train_loader)
    adapt_train_losses.append(avg_loss)

    #validation
    adapted_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in bc_test_loader:
            outputs = adapted_model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch['labels'].numpy())

    epoch_f1 = f1_score(all_labels, all_preds)
    adapt_val_f1s.append(epoch_f1)

    print(f"Epoch {epoch+1}/{ADAPT_EPOCHS} | "
          f"Loss: {avg_loss:.4f} | Val F1: {epoch_f1:.4f}")

    if epoch_f1 > best_f1:
        best_f1, best_epoch = epoch_f1, epoch + 1
        adapted_model.save_pretrained(f'{DATA_DIR}/blockchain/distilbert_adapted')
        tokeniser.save_pretrained(f'{DATA_DIR}/blockchain/model_checkpoint_adapted_final')

print(f"\nBest model: Epoch {best_epoch} — F1 {best_f1:.4f}")

In [ ]:
#load best adapted model
best_adapted = DBSC.from_pretrained(f'{DATA_DIR}/model_checkpoint_adapted_final')
best_adapted.to(device)
best_adapted.eval()

adapted_bc_preds, _ = bert_predict(best_adapted, bc_test['text'].values, tokeniser, device)

print("DEFINITIVE THREE-MODEL COMPARISON")
print()
print("General Email Test Set (n=1,600):")
print(f"  TF-IDF + LR:        F1={f1_score(y_test, tfidf_preds):.4f}  "
      f"ROC-AUC={roc_auc_score(y_test, tfidf_probs):.4f}")
print(f"  DistilBERT general: F1={f1_score(y_test, bert_preds):.4f}  "
      f"ROC-AUC={roc_auc_score(y_test, bert_probs):.4f}")
print()
print("Blockchain Test Set (n=162):")
print(f"  TF-IDF + LR:          F1={f1_score(y_bc, tfidf_bc_preds, zero_division=0):.4f}")
print(f"  DistilBERT general:   F1={f1_score(y_bc, bert_bc_preds, zero_division=0):.4f}")
print(f"  DistilBERT adapted:   F1={f1_score(y_bc, adapted_bc_preds, zero_division=0):.4f}")
print()
print(f"Domain gap (DistilBERT general): "
      f"{f1_score(y_test, bert_preds):.4f} -> "
      f"{f1_score(y_bc, bert_bc_preds, zero_division=0):.4f}")
print(f"Gap closed by adaptation: "
      f"{f1_score(y_bc, bert_bc_preds, zero_division=0):.4f} -> "
      f"{f1_score(y_bc, adapted_bc_preds, zero_division=0):.4f}")

In [ ]:
#Fig 4.5 — Domain-adapted confusion matrix
fig, ax = plt.subplots(figsize=(5,4))
cm = confusion_matrix(y_bc, adapted_bc_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate','Phishing'],
            yticklabels=['Legitimate','Phishing'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Figure 4.5: DistilBERT Domain-Adapted\nBlockchain Test Set (n=162)')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig4_5_adapted_confusion.png', dpi=150)
plt.show()

#Fig 4.6 — F1 comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(12,5))

gen_models = ['TF-IDF + LR', 'DistilBERT\n(general)']
gen_f1s = [f1_score(y_test, tfidf_preds), f1_score(y_test, bert_preds)]

bc_models = ['TF-IDF + LR', 'DistilBERT\n(general)', 'DistilBERT\n(adapted)']
bc_f1s = [
    f1_score(y_bc, tfidf_bc_preds, zero_division=0),
    f1_score(y_bc, bert_bc_preds, zero_division=0),
    f1_score(y_bc, adapted_bc_preds, zero_division=0),
]

bars1 = axes[0].bar(gen_models, gen_f1s,
                    color=['#2E75B6','#4A90D9'], alpha=0.85)
axes[0].set_ylim(0, 1.05); axes[0].set_ylabel('F1-Score')
axes[0].set_title('General Email Test Set')
for bar in bars1:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{bar.get_height():.4f}', ha='center', fontsize=10)

bars2 = axes[1].bar(bc_models, bc_f1s,
                    color=['#2E75B6','#4A90D9','#1A6B3C'], alpha=0.85)
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel('F1-Score')
axes[1].set_title('Blockchain Domain Test Set')
for bar in bars2:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{bar.get_height():.4f}', ha='center', fontsize=10)

fig.suptitle('Figure 4.6: F1-Score Comparison Across All Models and Test Sets',
             fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig4_6_f1_comparison.png', dpi=150)
plt.show()

#domain adaptation training curve
fig, ax1 = plt.subplots(figsize=(8,5))
ax2 = ax1.twinx()
epochs = list(range(1, ADAPT_EPOCHS+1))
ax1.plot(epochs, adapt_train_losses, 'b-o', label='Training Loss', linewidth=2)
ax2.plot(epochs, adapt_val_f1s, 'g-s', label='Validation F1', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss', color='blue')
ax2.set_ylabel('Validation F1', color='green')
ax1.set_title('Figure 4.4 (Domain Adaptation): Training Curve')
ax1.set_xticks(epochs)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='center right')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig4_4_domain_adaptation_curve.png', dpi=150)
plt.show()